In [5]:
import os
import json
import shutil
from glob import glob
from PIL import Image, ImageDraw
import random
import yaml

SRC_DIRS = ['./light', './night']
OUT_DIR = './original_dataset'
IMG_EXTS = ['.jpg', '.JPG', '.jpeg', '.JPEG', '.png', '.PNG', '.bmp', '.BMP']

for sub in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)

def get_all_images(src_dirs):
    images = []
    for src_dir in src_dirs:
        for ext in IMG_EXTS:
            images += glob(os.path.join(src_dir, f'*{ext}'))
    return images

def shape_to_yolo_bbox(shape, w, h):
    pts = shape['points']
    x1, y1 = pts[0]
    x2, y2 = pts[1]
    xmin, xmax = sorted([x1, x2])
    ymin, ymax = sorted([y1, y2])
    x_center = (xmin + xmax) / 2 / w
    y_center = (ymin + ymax) / 2 / h
    box_w = (xmax - xmin) / w
    box_h = (ymax - ymin) / h
    return x_center, y_center, box_w, box_h

def get_label_list(json_files):
    label_set = set()
    for jf in json_files:
        with open(jf, 'r', encoding='utf-8') as f:
            data = json.load(f)
            for s in data.get('shapes', []):
                label_set.add(s['label'])
    return sorted(list(label_set))

def main():
    images = get_all_images(SRC_DIRS)
    # 取得所有 json 標註檔案
    jsons = []
    for src_dir in SRC_DIRS:
        jsons += glob(os.path.join(src_dir, '*.json'))
    all_labels = get_label_list(jsons)
    label2id = {name: idx for idx, name in enumerate(all_labels)}
    print(f"共偵測到類別：{label2id}")

    samples = []
    for img_path in images:
        base = os.path.splitext(os.path.basename(img_path))[0]
        json_path = None
        for src_dir in SRC_DIRS:
            candidate = os.path.join(src_dir, base + '.json')
            if os.path.exists(candidate):
                # 檢查標註內容不為空
                with open(candidate, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    if len(data.get('shapes', [])) > 0:
                        json_path = candidate
                        samples.append((img_path, json_path))
                break  # 已找到標註，不用繼續檢查其它資料夾

    random.shuffle(samples)
    n_total = len(samples)
    n_train = int(n_total * 0.8)
    train_samples = samples[:n_train]
    val_samples = samples[n_train:]
    
    print(f"Train: {len(train_samples)}, Val: {len(val_samples)}")
    
    # 統計每個類別在 train/val 的出現張數
    class_train_count = {name: 0 for name in all_labels}
    class_val_count = {name: 0 for name in all_labels}

    def count_class(samples, counter_dict):
        for img_path, json_path in samples:
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            for shape in data.get('shapes', []):
                label = shape['label']
                if label in counter_dict:
                    counter_dict[label] += 1

    count_class(train_samples, class_train_count)
    count_class(val_samples, class_val_count)

    print('\n--- 類別在 train/val 的張數 ---')
    for name in all_labels:
        t = class_train_count[name]
        v = class_val_count[name]
        total = t + v
        tr = t / total if total else 0
        va = v / total if total else 0
        print(f"{name}: train={t}, val={v}, split比例 = {tr:.2%} / {va:.2%}")

    # YOLO 轉換與儲存
    img_label_dict = {}  # 方便後面預覽用
    for split_name, split_samples in zip(['train', 'val'], [train_samples, val_samples]):
        for img_path, json_path in split_samples:
            base = os.path.splitext(os.path.basename(img_path))[0]
            out_img = os.path.join(OUT_DIR, f'images/{split_name}', os.path.basename(img_path))
            out_label = os.path.join(OUT_DIR, f'labels/{split_name}', base + '.txt')
            shutil.copy2(img_path, out_img)

            yolo_lines = []
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            w, h = Image.open(img_path).size
            shapes = data.get('shapes', [])
            this_labels = []
            for shape in shapes:
                label = shape['label']
                if label not in label2id or len(shape['points']) != 2:
                    continue
                class_id = label2id[label]
                xc, yc, bw, bh = shape_to_yolo_bbox(shape, w, h)
                yolo_line = f"{class_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}"
                yolo_lines.append(yolo_line)
                this_labels.append((class_id, shape['points']))  # 存下 bbox 座標給預覽
            img_label_dict[out_img] = this_labels

            with open(out_label, 'w', encoding='utf-8') as f:
                f.write('\n'.join(yolo_lines))

    print("全部轉換與切分完成！")

    abs_train_path = os.path.abspath(os.path.join(OUT_DIR, 'images/train'))
    abs_val_path = os.path.abspath(os.path.join(OUT_DIR, 'images/val'))
    data_yaml = {
        'train': abs_train_path,
        'val': abs_val_path,
        'nc': len(all_labels),
        'names': all_labels
    }
    with open(os.path.join(OUT_DIR, 'data.yaml'), 'w', encoding='utf-8') as f:
        yaml.dump(data_yaml, f, allow_unicode=True)
    print('已產生 data.yaml (使用絕對路徑)')
    
    # ========= 預覽每個類別兩張圖 =========
    print('開始繪製每類別的隨機兩張圖片供檢查...')
    # 建立預覽圖片資料夾（改為工作目錄）
    preview_dir = os.path.join(os.getcwd(), 'preview_images')
    os.makedirs(preview_dir, exist_ok=True)
    # 對每一個類別，收集含該類別的所有圖片
    class_img_map = {i: [] for i in range(len(all_labels))}
    for img_path, bboxes in img_label_dict.items():
        for class_id, pts in bboxes:
            class_img_map[class_id].append(img_path)

    for class_id, img_list in class_img_map.items():
        sel_imgs = random.sample(img_list, min(2, len(img_list)))
        class_name = all_labels[class_id]
        for img_path in sel_imgs:
            img = Image.open(img_path).convert('RGB')
            draw = ImageDraw.Draw(img)
            # 重新讀出 bboxes
            for cid, pts in img_label_dict[img_path]:
                if cid == class_id:
                    x1, y1 = pts[0]
                    x2, y2 = pts[1]
                    draw.rectangle([x1, y1, x2, y2], outline='red', width=3)
            orig_base = os.path.splitext(os.path.basename(img_path))[0]
            save_path = os.path.join(preview_dir, f'{class_name}_{orig_base}.jpg')
            img.save(save_path)
            print(f"已儲存: {save_path}")

if __name__ == '__main__':
    main()


共偵測到類別：{'僅准右轉': 0, '右彎': 1, '右道封閉': 2, '圓環': 3, '學校': 4, '左彎': 5, '慢行': 6, '機慢車兩段式左轉': 7, '當心兒童': 8, '當心自行車': 9, '當心行人': 10, '禁止停車': 11, '禁止右轉': 12, '禁止喇叭': 13, '禁止大客車右轉': 14, '禁止大客車左轉': 15, '禁止大客車進入': 16, '禁止大貨車及大客車左轉': 17, '禁止大貨車及大客車進入': 18, '禁止大貨車右轉': 19, '禁止大貨車左右轉': 20, '禁止大貨車左轉': 21, '禁止大貨車進入': 22, '禁止左右轉': 23, '禁止左轉': 24, '禁止左轉及直行': 25, '禁止機車進入': 26, '禁止汽機車進入': 27, '禁止汽車右轉': 28, '禁止汽車左轉': 29, '禁止汽車進入': 30, '禁止腳踏車左轉': 31, '禁止臨時停車': 32, '禁止行人進入': 33, '禁止迴車': 34, '禁止連結車右轉': 35, '禁止連結車左轉': 36, '禁止連結車進入': 37, '禁止進入': 38, '讓路': 39, '路滑': 40, '連續彎路先向左彎': 41, '道路施工': 42, '道路顛簸': 43, '違規取締': 44, '險升坡': 45, '險降坡': 46}
Train: 2656, Val: 664

--- 類別在 train/val 的張數 ---
僅准右轉: train=16, val=6, split比例 = 72.73% / 27.27%
右彎: train=48, val=10, split比例 = 82.76% / 17.24%
右道封閉: train=16, val=2, split比例 = 88.89% / 11.11%
圓環: train=30, val=4, split比例 = 88.24% / 11.76%
學校: train=47, val=11, split比例 = 81.03% / 18.97%
左彎: train=48, val=16, split比例 = 75.00% / 25.00%
慢行: train=84, val=23, split比例 = 78.50% /

In [1]:
import os
import json
from glob import glob
from ultralytics import YOLO
from tqdm import tqdm

SRC_DIRS = ['./light', './night']
OUT_DIR = './predict'
WEIGHTS = './detect/weights/best.pt'
IMG_EXTS = ['.jpg', '.jpeg', '.png', '.bmp']

# 讀出類別名稱（模型要訓練過）
model = YOLO(WEIGHTS)
names = model.names  # 類別名稱列表，{0:'xxx', 1:'yyy', ...}

for src_dir in SRC_DIRS:
    sub_name = os.path.basename(src_dir)
    out_dir = os.path.join(OUT_DIR, sub_name)
    os.makedirs(out_dir, exist_ok=True)

    images = []
    for ext in IMG_EXTS:
        images += glob(os.path.join(src_dir, f'*{ext}'))

    print(f"Predicting {sub_name}, total images: {len(images)}")
    for img_path in tqdm(images):
        results = model.predict(img_path, conf=0.5, iou=0.5, device=0, verbose=False)
        boxes = results[0].boxes
        if len(boxes) > 0:
            shapes = []
            for box in boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().tolist()
                cls_id = int(box.cls[0].cpu().item())
                conf = float(box.conf[0].cpu().item())
                shapes.append({
                    "label": names[cls_id],  # 類別名稱
                    "points": [[x1, y1], [x2, y2]],
                    "confidence": conf,
                    "shape_type": "rectangle"
                })
            out_json = {
                "imagePath": os.path.basename(img_path),
                "shapes": shapes
            }
            # 存 json
            base = os.path.splitext(os.path.basename(img_path))[0]
            save_path = os.path.join(out_dir, base + '.json')
            with open(save_path, 'w', encoding='utf-8') as f:
                json.dump(out_json, f, ensure_ascii=False, indent=2)


Predicting light, total images: 17456


100%|██████████| 17456/17456 [06:11<00:00, 46.99it/s]


Predicting night, total images: 3910


100%|██████████| 3910/3910 [01:16<00:00, 51.16it/s]


In [2]:
import os
import json
import shutil
import pandas as pd
from glob import glob

def get_boxes_from_labelme(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    shapes = data.get("shapes", [])
    boxes = []
    for s in shapes:
        if s.get("shape_type") == "rectangle":
            label = s.get("label")
            (x1, y1), (x2, y2) = s["points"]
            xmin, ymin, xmax, ymax = min(x1, x2), min(y1, y2), max(x1, x2), max(y1, y2)
            boxes.append((label, xmin, ymin, xmax, ymax))
    return boxes, data.get("imagePath", None)

def iou(boxA, boxB):
    xA = max(boxA[1], boxB[1])
    yA = max(boxA[2], boxB[2])
    xB = min(boxA[3], boxB[3])
    yB = min(boxA[4], boxB[4])
    interW = max(0, xB - xA)
    interH = max(0, yB - yA)
    interArea = interW * interH
    boxAArea = (boxA[3]-boxA[1]) * (boxA[4]-boxA[2])
    boxBArea = (boxB[3]-boxB[1]) * (boxB[4]-boxB[2])
    iou = interArea / float(boxAArea + boxBArea - interArea + 1e-8)
    return iou

folders = ["light", "night"]
rows = []
all_TP = all_FP = all_FN = 0

os.makedirs("./result", exist_ok=True)

for folder in folders:
    gt_dir = f"./{folder}"
    pred_dir = f"./predict/{folder}"
    out_dir = f"./result/{folder}"
    os.makedirs(out_dir, exist_ok=True)

    gt_jsons = glob(os.path.join(gt_dir, "*.json"))
    TP = FP = FN = 0

    for gt_path in gt_jsons:
        basename = os.path.basename(gt_path)
        name, ext = os.path.splitext(basename)
        pred_path = os.path.join(pred_dir, basename)
        if not os.path.exists(pred_path):
            continue
        gt_boxes, img_name = get_boxes_from_labelme(gt_path)
        pred_boxes, _ = get_boxes_from_labelme(pred_path)

        matched_gt = set()
        matched_pred = set()

        for i, gt in enumerate(gt_boxes):
            for j, pred in enumerate(pred_boxes):
                if pred[0] == gt[0]:
                    if iou(gt, pred) >= 0.5:
                        matched_gt.add(i)
                        matched_pred.add(j)
        cur_TP = len(matched_gt)
        cur_FP = len(pred_boxes) - len(matched_pred)
        cur_FN = len(gt_boxes) - len(matched_gt)
        TP += cur_TP
        FP += cur_FP
        FN += cur_FN

        # 如果這張圖有 FP 或 FN
        if cur_FP > 0 or cur_FN > 0:
            # 複製原始圖片
            if img_name is not None:
                src_img_path = os.path.join(gt_dir, img_name)
                dst_img_path = os.path.join(out_dir, img_name)
                if os.path.exists(src_img_path):
                    shutil.copy2(src_img_path, dst_img_path)
            # 複製 gt json，加在檔名後
            shutil.copy2(gt_path, os.path.join(out_dir, f"{name}_gt{ext}"))
            # 複製 pred json，加在檔名後
            if os.path.exists(pred_path):
                shutil.copy2(pred_path, os.path.join(out_dir, f"{name}_pred{ext}"))

    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)
    F1 = 2 * precision * recall / (precision + recall + 1e-8)
    rows.append([folder, TP, FP, FN, precision, recall, F1])
    all_TP += TP
    all_FP += FP
    all_FN += FN

# ALL 場景加總
precision = all_TP / (all_TP + all_FP + 1e-8)
recall = all_TP / (all_TP + all_FN + 1e-8)
F1 = 2 * precision * recall / (precision + recall + 1e-8)
rows.append(["ALL", all_TP, all_FP, all_FN, precision, recall, F1])

df = pd.DataFrame(rows, columns=["scene", "TP", "FP", "FN", "precision", "recall", "F1"])
df.to_csv("./result/result.csv", index=False)
print("Done! 結果已輸出到 ./result/result.csv，並複製需要檢查的資料到 ./result/{light,night}/")


Done! 結果已輸出到 ./result/result.csv，並複製需要檢查的資料到 ./result/{light,night}/


In [3]:
import os
import json
from glob import glob
from PIL import Image, ImageDraw, ImageTk
import tkinter as tk

IMG_EXTS = ['.jpg', '.jpeg', '.png', '.bmp']

def find_images_with_ann(result_dirs):
    """找所有有 _gt.json 的圖片，回傳 {basename: (img_path, gt_json, pred_json)}"""
    all_img = {}
    for result_dir in result_dirs:
        for img_path in glob(os.path.join(result_dir, '*')):
            base, ext = os.path.splitext(os.path.basename(img_path))
            if ext.lower() not in IMG_EXTS:
                continue
            gt_json = os.path.join(result_dir, f"{base}_gt.json")
            pred_json = os.path.join(result_dir, f"{base}_pred.json")
            if os.path.exists(gt_json):
                all_img[os.path.join(result_dir, base)] = (img_path, gt_json, pred_json if os.path.exists(pred_json) else None)
    return all_img

def get_shapes(json_path):
    if not json_path or not os.path.exists(json_path):
        return []
    with open(json_path, encoding='utf-8') as f:
        js = json.load(f)
    return js.get('shapes', [])

class LabelmeResultGUI:
    def __init__(self, master, img_dict):
        self.master = master
        self.img_dict = list(img_dict.items())
        self.idx = 0
        self.img = None
        self.photo = None

        master.title("result 標註對比預覽")
        self.canvas = tk.Canvas(master, bg='#dddddd')
        self.canvas.pack(fill='both', expand=True)

        # 新增下方文字框分左右顯示 gt 與 pred 標註
        self.info_frame = tk.Frame(master)
        self.info_frame.pack(side='bottom', fill='x')

        self.gt_info = tk.Text(self.info_frame, height=8, width=48, font=('Arial', 14), wrap='none', state='disabled', bg='#fff3f3', fg='red')
        self.gt_info.pack(side='left', fill='both', expand=True)
        self.pred_info = tk.Text(self.info_frame, height=8, width=48, font=('Arial', 14), wrap='none', state='disabled', bg='#f3f7ff', fg='blue')
        self.pred_info.pack(side='right', fill='both', expand=True)

        self.info_label = tk.Label(master, text="", font=('Arial', 16))
        self.info_label.pack(side='bottom', fill='x')

        master.bind('<Configure>', self.on_resize)
        master.bind('<Left>', lambda event: self.prev_img())
        master.bind('<Right>', lambda event: self.next_img())

        self.show_img()

    def prev_img(self):
        if self.idx > 0:
            self.idx -= 1
            self.show_img()

    def next_img(self):
        if self.idx < len(self.img_dict) - 1:
            self.idx += 1
            self.show_img()

    def show_img(self):
        if not self.img_dict:
            self.canvas.delete("all")
            self.info_label.config(text="沒有圖片")
            self.gt_info.config(state='normal'); self.gt_info.delete(1.0, 'end'); self.gt_info.insert('end', '（無標記）'); self.gt_info.config(state='disabled')
            self.pred_info.config(state='normal'); self.pred_info.delete(1.0, 'end'); self.pred_info.insert('end', '（無標記）'); self.pred_info.config(state='disabled')
            return
        base, (img_path, gt_json, pred_json) = self.img_dict[self.idx]
        img = Image.open(img_path).convert('RGB')
        draw = ImageDraw.Draw(img)

        # 畫 gt 紅框
        gt_items = []
        for s in get_shapes(gt_json):
            pts = s['points']
            draw.rectangle([*pts[0], *pts[1]], outline='red', width=2)
            label = s.get('label', '')
            bbox = f"({pts[0][0]:.1f}, {pts[0][1]:.1f}) → ({pts[1][0]:.1f}, {pts[1][1]:.1f})"
            gt_items.append(f"{label}\nbox: {bbox}\n")

        # 畫 pred 藍框
        pred_items = []
        for s in get_shapes(pred_json):
            pts = s['points']
            draw.rectangle([*pts[0], *pts[1]], outline='blue', width=2)
            label = s.get('label', '')
            conf = s.get('confidence', None)
            bbox = f"({pts[0][0]:.1f}, {pts[0][1]:.1f}) → ({pts[1][0]:.1f}, {pts[1][1]:.1f})"
            if conf is not None:
                pred_items.append(f"{label} (score: {conf:.2f})\nbox: {bbox}\n")
            else:
                pred_items.append(f"{label}\nbox: {bbox}\n")

        self.img = img
        self.redraw_img()
        base_name = os.path.basename(img_path)
        self.info_label.config(text=f"{self.idx+1}/{len(self.img_dict)} : {base_name}")

        # 下方標記顯示
        self.gt_info.config(state='normal')
        self.gt_info.delete(1.0, 'end')
        self.gt_info.insert('end', '\n'.join(gt_items) if gt_items else '（無標記）')
        self.gt_info.config(state='disabled')

        self.pred_info.config(state='normal')
        self.pred_info.delete(1.0, 'end')
        self.pred_info.insert('end', '\n'.join(pred_items) if pred_items else '（無標記）')
        self.pred_info.config(state='disabled')

    def on_resize(self, event):
        if hasattr(self, 'img') and self.img is not None:
            self.redraw_img()

    def redraw_img(self):
        canvas_w = self.canvas.winfo_width()
        canvas_h = self.canvas.winfo_height()
        canvas_w = max(canvas_w, 400)
        canvas_h = max(canvas_h, 300)
        img = self.resize_keep_aspect(self.img, canvas_w, canvas_h)
        self.photo = ImageTk.PhotoImage(img)
        self.canvas.delete("all")
        x = (canvas_w - img.width) // 2
        y = (canvas_h - img.height) // 2
        self.canvas.create_image(x, y, anchor='nw', image=self.photo)

    def resize_keep_aspect(self, img, canvas_w, canvas_h):
        w, h = img.size
        scale = min(canvas_w / w, canvas_h / h)
        new_w = int(w * scale)
        new_h = int(h * scale)
        return img.resize((new_w, new_h), Image.LANCZOS)

if __name__ == '__main__':
    img_dict = find_images_with_ann(['./result/light', './result/night'])
    root = tk.Tk()
    root.attributes('-zoomed', True)
    gui = LabelmeResultGUI(root, img_dict)
    root.mainloop()


In [3]:
import os
import time
import gc
import socket
from pathlib import Path

from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from googleapiclient.errors import HttpError

from tqdm import tqdm

# ================== 設定 ==================
ROOT_FOLDER_ID = "1QxEVkh43a9CcaQV9Y3yy8eaml1jQhXpH"
LOCAL_ROOT = Path("/media/alex930307/TRANSCEND/GM_upload")

CLIENT_SECRET = "/home/alex930307/GM/client_secrets.json"
TOKEN_FILE = "/home/alex930307/GM/token.json"

SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]

CHUNK_SIZE = 1 * 1024 * 1024   # ⭐ 32MB（最穩）
MAX_RETRIES = 5
SLEEP_BETWEEN_REQUESTS = 0.1    # 防止 API throttle

LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

# ================== OAuth ==================
creds = None
if os.path.exists(TOKEN_FILE):
    creds = Credentials.from_authorized_user_file(TOKEN_FILE, SCOPES)

if not creds or not creds.valid:
    flow = InstalledAppFlow.from_client_secrets_file(CLIENT_SECRET, SCOPES)
    creds = flow.run_local_server(port=0)
    with open(TOKEN_FILE, "w") as f:
        f.write(creds.to_json())

service = build("drive", "v3", credentials=creds, cache_discovery=False)

# ================== 列出資料夾（遞迴） ==================
def list_folder(folder_id, parent_path):
    query = f"'{folder_id}' in parents and trashed=false"
    page_token = None

    while True:
        try:
            response = service.files().list(
                q=query,
                fields="nextPageToken, files(id, name, mimeType, size)",
                pageToken=page_token,
                pageSize=1000,
            ).execute()
        except HttpError as e:
            print("⚠️ list timeout，重試中…")
            time.sleep(2)
            continue

        for item in response.get("files", []):
            item_path = parent_path / item["name"]

            if item["mimeType"] == "application/vnd.google-apps.folder":
                item_path.mkdir(exist_ok=True)
                yield from list_folder(item["id"], item_path)
            else:
                yield item, item_path

        page_token = response.get("nextPageToken")
        if not page_token:
            break

# ================== 下載檔案 ==================
for idx, (file, save_path) in enumerate(list_folder(ROOT_FOLDER_ID, LOCAL_ROOT), 1):
    if save_path.exists():
        print(f"⏩ 已存在，跳過：{save_path}")
        continue

    print(f"\n⬇️ [{idx}] 下載：{save_path}")
    save_path.parent.mkdir(parents=True, exist_ok=True)

    request = service.files().get_media(fileId=file["id"])
    size = int(file.get("size", 0))

    with open(save_path, "wb") as fh, tqdm(
        total=size,
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
        desc=save_path.name,
    ) as pbar:

        downloader = MediaIoBaseDownload(
            fh, request, chunksize=CHUNK_SIZE
        )

        done = False
        retries = 0
        last = 0

        while not done:
            try:
                status, done = downloader.next_chunk()
                if status:
                    current = int(status.progress() * size)
                    pbar.update(current - last)
                    last = current
                retries = 0
            except (HttpError, TimeoutError, socket.timeout):
                retries += 1
                if retries > MAX_RETRIES:
                    print(f"❌ 放棄：{save_path}")
                    break
                wait = min(60, 2 ** retries)
                print(f"⚠️ timeout，{wait}s 後重試")
                time.sleep(wait)

    del downloader
    gc.collect()
    time.sleep(SLEEP_BETWEEN_REQUESTS)

print("\n🎉 全部資料夾下載完成")


⏩ 已存在，跳過：/media/alex930307/TRANSCEND/GM_upload/252/images/19.jpg
⏩ 已存在，跳過：/media/alex930307/TRANSCEND/GM_upload/252/images/20.jpg
⏩ 已存在，跳過：/media/alex930307/TRANSCEND/GM_upload/252/images/15.jpg
⏩ 已存在，跳過：/media/alex930307/TRANSCEND/GM_upload/252/images/14.jpg
⏩ 已存在，跳過：/media/alex930307/TRANSCEND/GM_upload/252/images/11.jpg
⏩ 已存在，跳過：/media/alex930307/TRANSCEND/GM_upload/252/images/8.jpg
⏩ 已存在，跳過：/media/alex930307/TRANSCEND/GM_upload/252/images/17.jpg
⏩ 已存在，跳過：/media/alex930307/TRANSCEND/GM_upload/252/images/13.jpg
⏩ 已存在，跳過：/media/alex930307/TRANSCEND/GM_upload/252/images/12.jpg
⏩ 已存在，跳過：/media/alex930307/TRANSCEND/GM_upload/252/images/18.jpg
⏩ 已存在，跳過：/media/alex930307/TRANSCEND/GM_upload/252/images/10.jpg
⏩ 已存在，跳過：/media/alex930307/TRANSCEND/GM_upload/252/images/9.jpg
⏩ 已存在，跳過：/media/alex930307/TRANSCEND/GM_upload/252/images/16.jpg

⬇️ [14] 下載：/media/alex930307/TRANSCEND/GM_upload/252/images/2.jpg


2.jpg: 100%|██████████| 394k/394k [00:01<00:00, 372kB/s]



⬇️ [15] 下載：/media/alex930307/TRANSCEND/GM_upload/252/images/6.jpg


6.jpg: 100%|██████████| 386k/386k [00:00<00:00, 448kB/s]



⬇️ [16] 下載：/media/alex930307/TRANSCEND/GM_upload/252/images/1.jpg


...


⬇️ [22] 下載：/media/alex930307/TRANSCEND/GM_upload/151/images/20.jpg


20.jpg: 100%|██████████| 488k/488k [00:00<00:00, 631kB/s]



⬇️ [23] 下載：/media/alex930307/TRANSCEND/GM_upload/151/images/19.jpg


19.jpg:   0%|          | 0.00/476k [00:00<?, ?B/s]


KeyboardInterrupt: 